In [1]:
# Clone repos + install
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .
!pip install -q transformers accelerate datasets zstandard tqdm

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Drive output dir for sweep results
!mkdir -p /content/drive/MyDrive/thesis_results/task02_percentile_sweep/opt-1.3b

# Verify all per-p scale files are present
!nvidia-smi
!ls -la /content/drive/MyDrive/thesis_results/act_percentiles/opt-1.3b/
!python -c "from smoothquant.fake_quant import quantize_model; print('smoothquant OK')"

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 97 (delta 29), reused 87 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 4.81 MiB | 22.08 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/llm-quantization-thesis
Cloning into 'smoothquant_repo'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 352 (delta 120), reused 90 (delta 90), pack-reused 183 (from 1)
Receiving objects: 100% (352/352), 6.80 MiB | 11.10 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
Mounted at /content/drive
Sun May  3 14:52:28 2026       
+---------------------------------------------------

In [3]:
%%writefile /content/percentile_smooth.py
"""Percentile-based smoothing for SmoothQuant Task 02 (OPT only).

Mirrors `smoothquant.smooth.smooth_lm` but replaces both activation- and
weight-side per-channel `max(|.|)` with per-channel `quantile(., p)`. When
`p == 1.0` it falls back to exact `max` so the baseline row of any sweep is
bit-identical to the upstream `smooth_lm`.
"""
import torch
import torch.nn as nn
from transformers.models.opt.modeling_opt import OPTDecoderLayer


@torch.no_grad()
def _per_channel_weight_stat(fcs, p_w, dtype):
    stacked = torch.cat([fc.weight.abs() for fc in fcs], dim=0)
    if p_w >= 1.0:
        scales = stacked.max(dim=0).values
    else:
        scales = stacked.float().quantile(p_w, dim=0).to(stacked.dtype)
    return scales.to(dtype).clamp_(min=1e-5)


@torch.no_grad()
def smooth_ln_fcs_pct(ln, fcs, act_scales, alpha=0.5, p_w=1.0):
    if not isinstance(fcs, list):
        fcs = [fcs]
    assert isinstance(ln, nn.LayerNorm)
    for fc in fcs:
        assert isinstance(fc, nn.Linear)
        assert ln.weight.numel() == fc.in_features == act_scales.numel()

    device, dtype = fcs[0].weight.device, fcs[0].weight.dtype
    act_scales = act_scales.to(device=device, dtype=dtype).clamp_(min=1e-5)
    weight_scales = _per_channel_weight_stat(fcs, p_w, dtype).to(device)

    scales = (
        (act_scales.pow(alpha) / weight_scales.pow(1 - alpha))
        .clamp(min=1e-5)
        .to(device)
        .to(dtype)
    )

    ln.weight.div_(scales)
    ln.bias.div_(scales)
    for fc in fcs:
        fc.weight.mul_(scales.view(1, -1))


@torch.no_grad()
def smooth_lm_pct(model, scales, alpha=0.5, p_w=1.0):
    for name, module in model.named_modules():
        if isinstance(module, OPTDecoderLayer):
            attn_ln = module.self_attn_layer_norm
            qkv = [
                module.self_attn.q_proj,
                module.self_attn.k_proj,
                module.self_attn.v_proj,
            ]
            qkv_input_scales = scales[name + ".self_attn.q_proj"]
            smooth_ln_fcs_pct(attn_ln, qkv, qkv_input_scales, alpha, p_w)

            ffn_ln = module.final_layer_norm
            fc1 = module.fc1
            fc1_input_scales = scales[name + ".fc1"]
            smooth_ln_fcs_pct(ffn_ln, fc1, fc1_input_scales, alpha, p_w)

Writing /content/percentile_smooth.py


In [4]:
import sys
sys.path.insert(0, "/content/llm-quantization-thesis/smoothquant_repo")
sys.path.insert(0, "/content")  # for the %%writefile'd percentile_smooth.py

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from smoothquant.fake_quant import quantize_model
from percentile_smooth import smooth_lm_pct
from datasets import load_dataset
import json, os, time, tqdm

MODEL = "facebook/opt-1.3b"
PCT_DIR = "/content/drive/MyDrive/thesis_results/act_percentiles/opt-1.3b"
SAVE_DIR = "/content/drive/MyDrive/thesis_results/task02_percentile_sweep/opt-1.3b"

# Sweep grid. Each p maps to its own .pt file under PCT_DIR.
PERCENTILES = [1.0, 0.999, 0.995, 0.99, 0.95, 0.90]
ALPHAS = [0.5, 0.7, 0.9]

# Scheme C — per-channel W + per-token A (Task 01 winner).
WEIGHT_QUANT = "per_channel"
ACT_QUANT = "per_token"


class Evaluator:
    def __init__(self, dataset, tokenizer, device, n_samples=40):
        self.dataset = tokenizer(
            "\n\n".join(dataset["text"]), return_tensors="pt"
        ).input_ids.to(device)
        self.n_samples = n_samples

    @torch.no_grad()
    def evaluate(self, model):
        model.eval()
        nlls = []
        n = self.n_samples if self.n_samples else self.dataset.size(1) // 2048
        for i in tqdm.tqdm(range(n), desc="Evaluating"):
            batch = self.dataset[:, (i * 2048):((i + 1) * 2048)].to(model.device)
            lm_logits = model(batch).logits
            shift_logits = lm_logits[:, :-1, :].contiguous().float()
            shift_labels = self.dataset[:, (i * 2048):((i + 1) * 2048)][:, 1:]
            loss = nn.CrossEntropyLoss()(
                shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)
            )
            nlls.append(loss.float() * 2048)
        return torch.exp(torch.stack(nlls).sum() / (n * 2048))


# --- Load tokenizer + dataset once; lazy-load each per-p file when needed ---
print("Loading tokenizer + dataset...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
evaluator = Evaluator(dataset, tokenizer, "cuda")


def scales_path_for(p):
    return os.path.join(PCT_DIR, f"p{p:g}.pt")


# Sanity-load every required scale file before starting the sweep, so we fail
# fast if any are missing rather than 30 minutes in.
for p in PERCENTILES:
    path = scales_path_for(p)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing scale file for p={p}: {path}")
print(f"All {len(PERCENTILES)} per-p scale files present.")


all_results = []
total_runs = len(PERCENTILES) * len(ALPHAS)
run_num = 0

for p in PERCENTILES:
    print(f"\n[loading scales] p={p:g} from {scales_path_for(p)}")
    scales_for_p = torch.load(scales_path_for(p))

    for alpha in ALPHAS:
        run_num += 1
        config_label = f"C-pct{p:g}-a{alpha}"
        print(f"\n{'='*60}")
        print(f"  Run {run_num}/{total_runs}: {config_label}")
        print(f"  Scheme C (per_channel W, per_token A)")
        print(f"  p={p}, alpha={alpha}")
        print(f"{'='*60}")

        start = time.time()

        model = AutoModelForCausalLM.from_pretrained(
            MODEL, torch_dtype=torch.bfloat16, device_map="auto"
        )

        smooth_lm_pct(model, scales_for_p, alpha=alpha, p_w=p)

        model = quantize_model(
            model,
            weight_quant=WEIGHT_QUANT,
            act_quant=ACT_QUANT,
            quantize_bmm_input=True,
        )

        ppl = evaluator.evaluate(model)
        elapsed = time.time() - start
        ppl_val = ppl.item()

        result = {
            "config_label": config_label,
            "model": MODEL,
            "scheme": "C",
            "weight_quant": WEIGHT_QUANT,
            "act_quant": ACT_QUANT,
            "p": p,
            "alpha": alpha,
            "wikitext2_ppl": round(ppl_val, 4),
            "duration_seconds": round(elapsed, 1),
        }
        all_results.append(result)

        fname = f"opt-1.3b_C_p{p:g}_a{alpha}.json"
        with open(os.path.join(SAVE_DIR, fname), "w") as f:
            json.dump(result, f, indent=2)

        print(f">>> {config_label}: PPL = {ppl_val:.4f} ({elapsed:.0f}s)")

        del model
        torch.cuda.empty_cache()

    del scales_for_p

# --- Save combined results ---
with open(os.path.join(SAVE_DIR, "all_results.json"), "w") as f:
    json.dump(all_results, f, indent=2)

Loading tokenizer + dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Evaluating: 100%|██████████| 40/40 [00:06<00:00,  6.24it/s]


>>> C-pct0.9-a0.7: PPL = 14.6272 (56s)

  Run 18/18: C-pct0.9-a0.9
  Scheme C (per_channel W, per_token A)
  p=0.9, alpha=0.9


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Evaluating: 100%|██████████| 40/40 [00:06<00:00,  6.25it/s]


>>> C-pct0.9-a0.9: PPL = 15.8997 (56s)


In [5]:
print(f"\n{'='*70}")
print(f"  PERCENTILE SWEEP — OPT-1.3B, Scheme C")
print(f"{'='*70}")
print(f"\n{'p':>8} " + " ".join(f"{'alpha=' + str(a):>12}" for a in ALPHAS))
print("-" * (10 + 13 * len(ALPHAS)))

by_pa = {(r["p"], r["alpha"]): r["wikitext2_ppl"] for r in all_results}
for p in PERCENTILES:
    row = f"{p:>8g} "
    for a in ALPHAS:
        ppl = by_pa.get((p, a))
        row += f" {ppl:>12.4f}" if ppl is not None else f" {'--':>12}"
    print(row)

best = min(all_results, key=lambda r: r["wikitext2_ppl"])
print(f"\nBest: p={best['p']:g}, alpha={best['alpha']}, PPL={best['wikitext2_ppl']:.4f}")

max_at_best_alpha = next(
    r for r in all_results if r["p"] == 1.0 and r["alpha"] == best["alpha"]
)
delta = max_at_best_alpha["wikitext2_ppl"] - best["wikitext2_ppl"]
print(
    f"vs. max-smoothing at alpha={best['alpha']}: "
    f"PPL {max_at_best_alpha['wikitext2_ppl']:.4f} → {best['wikitext2_ppl']:.4f} "
    f"(delta {delta:+.4f}, positive = percentile wins)"
)


  PERCENTILE SWEEP — OPT-1.3B, Scheme C

       p    alpha=0.5    alpha=0.7    alpha=0.9
-------------------------------------------------
       1       14.6621      14.6666      14.6130
   0.999       14.7363      14.6291      14.6167
   0.995       14.6883      14.6349      14.7110
    0.99       14.6907      14.6898      14.8002
    0.95       14.6790      14.6968      15.1264
     0.9       14.6793      14.6272      15.8997

Best: p=1, alpha=0.9, PPL=14.6130
vs. max-smoothing at alpha=0.9: PPL 14.6130 → 14.6130 (delta +0.0000, positive = percentile wins)
